# backprop-pop-outgrad-loop — ex3: instrument reverse-pass with per-node max|grad_out| trace

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backprop-pop-outgrad-loop`. Running the final beacon cell reports progress against the `Backprop: backprop pop-outgrad loop` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: backprop pop-outgrad loop` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backprop-pop-outgrad-loop`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backprop-pop-outgrad-loop"
DD_SUBTOPIC = "Backprop: backprop pop-outgrad loop"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Per-node max|grad| trace — vanishing/exploding diagnostic

Ex1 ran the reverse-pass driver; ex2 counted per-leaf accumulations.
The deepening move tracks the L∞ MAGNITUDE of each `grad_out` as it
flows back — a fingerprint of vanishing-gradient (values → 0 deep in
the graph) or exploding-gradient (values → ∞) pathologies.

```python
def backprop_traced(end_node, end_grad, sorted_graph, back_funcs):
    grads = {id(end_node): end_grad}
    trace = []  # [(node_id, max_abs_grad_out), ...] in pop order
    for node in sorted_graph:
        if id(node) not in grads: continue
        grad_out = grads.pop(id(node))
        trace.append((id(node), float(grad_out.abs().max())))
        # ... rest of driver as ex1 ...
    return trace
```

**Why max|.| not norm.** L∞ catches a SINGLE explosive element — one
rogue activation that's about to overflow on the next forward pass.
L2 averages it out. For health monitoring, the worst element is the
right signal.

**Trace order matches reverse-pass order.** The first entry is
`end_node`; the last entries are leaves. A monotone-decreasing trace
is the vanishing signature; an increasing one is exploding. Most real
graphs show a noisy mix — but a clean monotone pattern over 50+ layers
is what RNN tutorials famously visualize.

### Exercise 3 — instrument reverse-pass with per-node max|grad_out| trace

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the reverse pass by recording the L∞ magnitude (`grad_out.abs().max().item()`) for every node as it's popped, producing a trace list that fingerprints vanishing/exploding gradient pathologies.
> Keywords: trace, max-abs, vanishing-gradient, exploding-gradient, diagnostic
> ```

**KCs targeted:** `backprop-pop-outgrad-loop`, `linf-grad-trace-per-node`

Implement `ex3_backprop_traced(end_node, end_grad, sorted_graph, back_funcs)` — the same reverse-pass driver as ex1, but also return a `trace` list capturing the L∞ magnitude of each node's popped `grad_out`.

Signature:

```python
def ex3_backprop_traced(end_node, end_grad, sorted_graph, back_funcs):
    ...
    return trace  # list[tuple[int, float]]
```

Semantics:
- `trace` is a list of `(id(node), float(grad_out.abs().max()))` entries, appended IMMEDIATELY AFTER popping `grad_out` from the accumulator and BEFORE dispatching to back_fns.
- Only nodes that ACTUALLY get popped (i.e. have a grad routed to them) appear in the trace. Skipped nodes (no entry in `grads`) are absent.
- Order: matches the reverse-pass walk through `sorted_graph` — `end_node` typically first, leaves last.

Same three reverse-pass invariants as ex1:
1. POP (don't peek) the entry in `grads`.
2. ACCUMULATE (don't overwrite) — `+=` per parent.
3. LEAVES write `.grad`; non-leaves stay in the `grads` dict.

Mutate `.grad` on leaves in place; return only `trace`.

In [ ]:
def ex3_backprop_traced(end_node, end_grad, sorted_graph, back_funcs):
    grads = {id(end_node): end_grad}
    trace = []
    for node in sorted_graph:
        if id(node) not in grads:
            continue
        grad_out = grads.pop(id(node))
        # Record L\u221E magnitude AFTER pop, BEFORE dispatch — this is what
        # was routed to this node, including any diamond-DAG accumulation.
        trace.append((id(node), float(grad_out.abs().max().item())))
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp
    return trace


<details><summary>Solution</summary>

```python
def ex3_backprop_traced(end_node, end_grad, sorted_graph, back_funcs):
    grads = {id(end_node): end_grad}
    trace = []
    for node in sorted_graph:
        if id(node) not in grads:
            continue
        grad_out = grads.pop(id(node))
        # Record L\u221E magnitude AFTER pop, BEFORE dispatch — this is what
        # was routed to this node, including any diamond-DAG accumulation.
        trace.append((id(node), float(grad_out.abs().max().item())))
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp
    return trace
```

**Record AFTER pop, BEFORE dispatch.** Recording before pop would miss the accumulation step (diamond DAG: parent gets contributions from multiple children; the magnitude isn't final until the pop). Recording after dispatch would conflate the parent's incoming grad with the post-back_fn grad, which is a DIFFERENT quantity.

**L∞ catches the worst element.** A grad of shape `(1024,)` with 1023 zeros and one `1e10` is exploding in real terms; L2 averages it to `1e10/sqrt(1024) ≈ 3.1e8` which understates the danger. Max-abs is the right alarm signal.

**`.item()` for the float cast.** The trace must be JSON-serializable for downstream logging. Python floats (not 0-D tensors) achieve that.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()